In [ ]:
DATA_PATH = "../data/processed/s03_filter.parquet"

import pandas as pd
import numpy as np

df = pd.read_parquet(DATA_PATH)

print(f"Dataset size: {len(df):,}")
print(f"Unique products (parent_asin): {df['parent_asin'].nunique():,}")
print(df.columns.tolist())
df.head()

In [ ]:
import numpy as np

# -----------------------------
# label (binary classification) - already computed upstream as is_helpful
# -----------------------------
df["label"] = df["is_helpful"].astype(int)

# -----------------------------
# combined_text is already built upstream (title + review text, [SEP]-joined)
# -----------------------------
df["combined_text"] = df["combined_text"].fillna("")

# -----------------------------
# Tokenize for DistilBERT
# -----------------------------
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

encodings = tokenizer(
    df["combined_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

# -----------------------------
# numeric features matrix for MLP
# -----------------------------
numeric_features = df[
    [
        "rating",
        "review_length",
        "title_length",
        "exclamation_mark",
        "question_mark",
        "image_bucket",
        "is_verified",
        "title_to_text_ratio",
    ]
].values.astype(np.float32)

# label
labels = df["label"].values

In [7]:
# Train-test split
from sklearn.model_selection import train_test_split

############### FIX THIS LATER################
# i did tokenizing before the split
##############################################

# 70 train 15 val 15 test
# train vs temp
train_idx, temp_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.3,
    stratify=labels,
    random_state=42
)

# val vs test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=labels[temp_idx],
    random_state=42
)

In [8]:
# text encodings
train_encodings = {k: v[train_idx] for k, v in encodings.items()}
val_encodings   = {k: v[val_idx] for k, v in encodings.items()}
test_encodings  = {k: v[test_idx] for k, v in encodings.items()}

# numeric features
train_num = numeric_features[train_idx]
val_num   = numeric_features[val_idx]
test_num  = numeric_features[test_idx]

# standardize numerics
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_num = scaler.fit_transform(train_num)
val_num   = scaler.transform(val_num)
test_num  = scaler.transform(test_num)

# labels
train_labels = labels[train_idx]
val_labels   = labels[val_idx]
test_labels  = labels[test_idx]

In [9]:
import torch
from torch.utils.data import Dataset

class ReviewDataset(Dataset):
    def __init__(self, encodings, numeric_features, labels):
        self.encodings = encodings
        self.numeric = torch.tensor(numeric_features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["numeric"] = self.numeric[idx]
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

In [10]:
train_dataset = ReviewDataset(train_encodings, train_num, train_labels)
val_dataset   = ReviewDataset(val_encodings, val_num, val_labels)
test_dataset  = ReviewDataset(test_encodings, test_num, test_labels)

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [ ]:
print(len(train_dataset), len(val_dataset), len(test_dataset))

In [15]:
import torch
import torch.nn as nn
from transformers import DistilBertModel

class DistilBertMLPFusion(nn.Module):
    def __init__(
        self,
        bert_model_name="distilbert-base-uncased",
        numeric_dim=8,
        num_classes=2,
        text_dropout=0.3,
        fusion_hidden_dim=128
    ):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained(bert_model_name)
        bert_hidden_size = self.bert.config.hidden_size

        self.numeric_branch = nn.Sequential(
            nn.Linear(numeric_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(bert_hidden_size + 32, fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(text_dropout),
            nn.Linear(fusion_hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, numeric_features):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        numeric_out = self.numeric_branch(numeric_features)

        fused = torch.cat([cls_embedding, numeric_out], dim=1)
        logits = self.classifier(fused)
        return logits

2026-04-10 16:41:13.112381: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-10 16:41:13.112470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-10 16:41:13.663715: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-10 16:41:14.742369: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-10 16:41:17.975101: W tensorflow/compiler/tf2

In [ ]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = DistilBertMLPFusion(
    bert_model_name="distilbert-base-uncased",
    numeric_dim=train_num.shape[1],
    num_classes=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

In [20]:
# Training function

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    return avg_loss, acc, f1, precision, recall


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)

    return avg_loss, acc, f1, precision, recall

In [ ]:
# training loop

num_epochs = 3
best_val_f1 = 0
best_model_state = None

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1, train_prec, train_rec = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f} | Precision: {train_prec:.4f} | Recall: {train_rec:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | Precision: {val_prec:.4f} | Recall: {val_rec:.4f}")
    print("-" * 80)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

In [ ]:
# verify on test

if best_model_state is not None:
    model.load_state_dict(best_model_state)

test_loss, test_acc, test_f1, test_prec, test_rec = evaluate(
    model, test_loader, criterion, device
)

print("Best Validation F1:", best_val_f1)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.4f}")
print(f"Test F1:   {test_f1:.4f}")
print(f"Test Prec: {test_prec:.4f}")
print(f"Test Rec:  {test_rec:.4f}")

In [ ]:
# classification report

from sklearn.metrics import classification_report, confusion_matrix

@torch.no_grad()
def get_predictions(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_features = batch["numeric"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_features=numeric_features
        )

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)

y_true, y_pred = get_predictions(model, test_loader, device)

print(classification_report(y_true, y_pred, digits=4))
print(confusion_matrix(y_true, y_pred))